# Data Sources:

Based of the amazing work you did in the Movie Industry you've been recruited to the NBA! You are working as the VP of Analytics that helps support a head scout, Mr. Rooney, for the worst team in the NBA probably the Wizards. Mr. Rooney just heard about Data Science and thinks it can solve all the team's problems!!! He wants you to figure out a way to find players that are high performing but maybe not highly paid that you can steal to get the team to the playoffs! 

In this document you will work through a similar process that we did in class with the NBA data (NBA_Perf_22 and nba_salaries_22), merging them together. This is from 22-23 season, feel free to update to 2023-24 season if you want.

https://www.basketball-reference.com/leagues/NBA_2024_totals.html # reference for performance data

https://www.basketball-reference.com/contracts/players.html # reference for salary data

Details: 

- Determine a way to use clustering to estimate based on performance if 
players are under or over paid, generally. 

- Then select players you believe would be best for your team and explain why. Do so in three categories: 
    * Examples that are not good choices (3 or 4) 
    * Several options that are good choices (3 or 4)
    * Several options that could work, assuming you can't get the players in the good category (3 or 4)

- You will decide the cutoffs for each category, so you should be able to explain why you chose them.

- Provide a well commented and clean report of your findings in a separate notebook that can be presented to Mr. Rooney, keeping in mind he doesn't understand...anything. Include a rationale for variables you included in the model, details on your approach and a overview of the results with supporting visualizations. 


Hints:

- Salary is the variable you are trying to understand 
- When interpreting you might want to use graphs that include variables that are the most correlated with Salary
- You'll need to scale the variables before performing the clustering
- Be specific about why you selected the players that you did, more detail is better
- Use good coding practices, comment heavily, indent, don't use for loops unless totally necessary and create modular sections that align with some outcome. If necessary create more than one script,list/load libraries at the top and don't include libraries that aren't used. 
- Be careful for non-traditional characters in the players names, certain graphs won't work when these characters are included.


# Clustering Lab

### Import Libraries and Load Data

In [35]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

performance = pd.read_csv('../data/NBA_Perf_22.csv',encoding='latin1')
performance["Player"] = performance["Player"].str.replace("?", "ć")
salary = pd.read_csv('../data/nba_salaries_22.csv',encoding='utf-8-sig',engine='python')
salary['Salary'] = (
    salary['Salary']
      .str.replace(r'[\$,]', '', regex=True)
      .str.strip()
      .astype(float)
)

### Drop NAs, duplicates, and filter data

In [36]:
# Drop NAs
performance = performance.dropna()

# 1. Identify which columns are “percent” stats vs. pure counts
num_cols = performance.select_dtypes(include="number").columns

# 2. Group & sum everything, yielding one row per Player
performance = (
    performance
      .groupby("Player")[num_cols]
      .sum()
      .reset_index()
)

performance_norm = performance.copy()

In [37]:
# Drop the columns that are not needed
performance = performance.drop(columns=['Player','Age','FGA','3PA','2PA','FTA'])

# Exclude players who don't really play
def filter_played(df):
    df = df[df['MP'] > 10]
    df = df[df['G']>20]
    return df
performance = filter_played(performance)

### Standardize the variables

In [38]:
# Save the column and the index of the dataframe
col = performance.columns
index = performance.index

# now `agg` has exactly one row per Player  
# you can reset_index and carry on with scaling & k-Means:
#performance = agg.reset_index()

# Standardize the data
performance = performance.to_numpy()
performance_mean = np.mean(performance, axis=0)
performance_std = np.std(performance, axis=0)
performance=(performance-performance_mean)/performance_std
performance = pd.DataFrame(performance, columns= col, index=index)

### Run the clustering algo with your best guess for K

In [39]:
# Select Feature Data
featured_columns = ['PTS','eFG%','3P']
cluster_per = performance[featured_columns]
# Run the clustering algorithm with my best guess for K=3
kmeans_obj_performance = KMeans(n_clusters=3, random_state=1).fit(cluster_per)

### View the results

In [40]:
print(f'The cluster centers are {kmeans_obj_performance.cluster_centers_}')
print(f'The labels are {kmeans_obj_performance.labels_}')
print(f'The inertia is {kmeans_obj_performance.inertia_}')

The cluster centers are [[-0.23854117 -0.42775012 -0.20936511]
 [ 0.49984709  2.13041955  0.33348544]
 [ 3.1848397   2.3515407   3.08174706]]
The labels are [0 1 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 1 2 2 0 2 0 0 1 0 0 0 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0
 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 2 0 0 0 0 2 0 0 0 0 0 0
 2 0 2 0 0 0 1 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0
 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 1 0 2 0 0 0 0 0 0
 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 2 1 0 2 0 0 0 1 0 1 2 1
 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 2 0 0 0 0 0 0
 0 1 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1
 0 0 1 0 0 0 0 0 0 1 0 0 0 2 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0
 0 0 2 0 0 1 0 1 0 0 0 0 0 0 0 0 0 1 1 2 0 0 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 1 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 1 0 1 0 0 0 2 0 0 0 0 0 1 0 0 0 0 0 0 0
 0 0 0]
The inerti

### Create a visualization of the results with 2 or 3 variables that you think will best differentiate the clusters

In [41]:
fig = px.scatter_3d(cluster_per, x="PTS", y="eFG%", z="3P", color=kmeans_obj_performance.labels_, title="Points vs. eFG% vs. 3-Point Made for player performance")
fig.show(renderer="vscode")

- I pick PTS as it is the most direct contributor to a player's contribution. 
- I chose eFG% because it takes the positional difference of players into account. Centers tend to have a higher percentage as they attack close to the rim, but not necessarily eFG%, as it is calculated using both 2PT and 3PT shots.
- 3-point made is also a good indicator of performance. NBA nowadays is growing to be more positionless and 3PT demanding.

### Evaluate the quality of the clustering using total variance explained and silhouette scores

In [42]:
#Total variance
X = cluster_per.values  # or clust_performance.to_numpy()

# 1. Total Sum of Squares (TSS)
tss = np.sum((X - np.mean(X, axis=0))**2)

# 2. Between‑cluster SS (BSS) = TSS – WSS (where WSS is inertia_)
wss = kmeans_obj_performance.inertia_
bss = tss - wss

# 3. Proportion of variance explained
var_explained = bss / tss

# 4. Silhouette score (single global value)
sil_score = silhouette_score(X, kmeans_obj_performance.labels_)

# 5. Print nicely
print(f'Total variance explained: {var_explained:.2%}')
print(f'Silhouette score: {sil_score:.3f}')

Total variance explained: 64.27%
Silhouette score: 0.594


### Determine the ideal number of clusters using the elbow method and the silhouette coefficient

In [43]:
# elbow method
wcss = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=1)
    km.fit(X)
    wcss.append(km.inertia_)
elbow_df = pd.DataFrame({"k": range(1,11), "wcss": wcss})

# Compute silhouette scores for k = 2…10
sil_scores = []
ks = range(2, 11)
for k in ks:
    km = KMeans(n_clusters=k, random_state=1)
    labels = km.fit_predict(X)
    sil_scores.append(silhouette_score(X, labels))
sil_df = pd.DataFrame({"k": list(ks), "silhouette": sil_scores})

# Determine best k by silhouette
best_sil_k = sil_df.loc[sil_df["silhouette"].idxmax(), "k"]
print(f"Best number of clusters by silhouette: {best_sil_k}")

# Plot elbow
fig1 = px.line(
    elbow_df, x="k", y="wcss", 
    title="Elbow Method"
)
fig1.update_layout(xaxis=dict(dtick=1))
fig1.show(renderer="vscode")

# Plot silhouette
fig2 = px.line(
    sil_df, x="k", y="silhouette", 
    title="Silhouette Scores"
)
fig2.update_layout(xaxis=dict(dtick=1))
fig2.show(renderer="vscode")


Best number of clusters by silhouette: 2


On the elbow chart, we can see a sharp drop in WCSS (the within-cluster sum of squares) when we add 1 more cluster to the model from k=1. In general, adding clusters always causes over-fitting, so the goal is to have the least amount of clusters as possible while explaining the variance. In this case, the "elbow" of the chart is at k=2, meaning that when we have 2 clusters, the WCSS is relatively low and the model stays away from over-fitting.

### Use the recommended number of cluster (assuming it's different) to retrain your model and visualize the results

In [44]:
kmeans_obj_performance_best = KMeans(n_clusters=2, random_state=1).fit(cluster_per)
performance['cluster'] = kmeans_obj_performance_best.labels_
fig = px.scatter_3d(performance, x="eFG%", y="PTS", z="3P", color='cluster', title="eFG% vs. PTS vs. 3-point made votes for player performance")
fig.show(renderer="vscode")

### Once again evaluate the quality of the clustering using total variance explained and silhouette scores

In [45]:
#Once again evaluate the quality of the clustering using total variance explained and silhouette scores
#Total variance

# Suppose `clust_performance` is your DataFrame of features:
X = cluster_per.values  # or clust_performance.to_numpy()

# 1. Total Sum of Squares (TSS)
tss = np.sum((X - np.mean(X, axis=0))**2)

# 2. Between‑cluster SS (BSS) = TSS – WSS (where WSS is inertia_)
wss = kmeans_obj_performance_best.inertia_
bss = tss - wss

# 3. Proportion of variance explained
var_explained = bss / tss

# 4. Silhouette score (single global value)
sil_score = silhouette_score(X, kmeans_obj_performance_best.labels_)

# 5. Print nicely
print(f'Total variance explained: {var_explained:.2%}')
print(f'Silhouette score: {sil_score:.3f}')


Total variance explained: 50.09%
Silhouette score: 0.637


### Use the model to select players for Mr. Rooney to consider

In [46]:
performance['Player'] = performance_norm.loc[index, 'Player'].values

In [47]:
import unicodedata
def normalize_name(name):
    name = str(name)
    nfkd = unicodedata.normalize("NFKD", name)
    stripped = "".join(c for c in nfkd if not unicodedata.combining(c))
    return stripped.lower().strip()

# 2. Create normalized‐name columns
performance['Player_norm'] = performance['Player'].map(normalize_name)
salary['Player_norm'] = salary['Player'].map(normalize_name)

# 3. Merge salary into performance
performance = performance.merge(
    salary[['Player_norm','Salary']],
    on='Player_norm',
    how='left'
)

# 4. Fill any missing salaries
#median_sal = salary['Salary'].median()
#performance['Salary'] = performance['Salary'].fillna(median_sal)
performance= performance.dropna()


# 5. 3D scatter with marker size ∝ Salary
fig = px.scatter_3d(
    performance,
    x="eFG%",
    y="PTS",
    z="3P",
    color="cluster",
    size="Salary",
    size_max=12,
    hover_data=["Player","Salary"],
    title="3D Performance Clusters (marker size ∝ salary)"
)
fig.show(renderer="vscode")

In [48]:
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd

# 0. Assume `performance` has your raw PTS, eFG%, 3P, and Salary columns.

# 1. Cluster on the 3 performance stats
kmeans = KMeans(n_clusters=3, random_state=1, algorithm="lloyd")
performance["cluster"] = kmeans.fit_predict(
    performance[["PTS","eFG%","3P"]]
)

# 2. Prepare the tree DataFrame
tree_data = performance[["Salary","cluster"]].copy()
# If you want to include the raw stats as well:
# tree_data = performance[["Salary","PTS","eFG%","3P","cluster"]].copy()

# 3. Convert cluster to categorical (and dummies)
tree_data["cluster"] = tree_data["cluster"].astype("category")
tree_data = pd.get_dummies(tree_data, columns=["cluster"], drop_first=True)

# 4. Train/test split
X = tree_data.drop(columns=["Salary"])
y = tree_data["Salary"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1
)

# 5. Fit the Decision Tree Regressor
dt_reg = DecisionTreeRegressor(max_depth=4, random_state=1)
dt_reg.fit(X_train, y_train)

# 6. Predict & evaluate
y_pred = dt_reg.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Test RMSE: {rmse:.0f}")
print(f"Test R²:   {r2_score(y_test, y_pred):.3f}")

# 7. Compute residuals on the full dataset (re-fit to all data)
dt_full = DecisionTreeRegressor(max_depth=4, random_state=1)
dt_full.fit(X, y)
performance["salary_pred"] = dt_full.predict(X)
performance["residual"]      = performance["Salary"] - performance["salary_pred"]

# 1. Define a dynamic 5% threshold per player
threshold = 0.05 * performance["salary_pred"]

# 2. Build conditions
conds = [
    performance["residual"] <= -threshold,   # actual << predicted → underpaid
    performance["residual"] >=  threshold    # actual >> predicted → overpaid
]
choices = ["underpaid", "overpaid"]

# 3. Assign “fair” by default
performance["pay_category_dt"] = np.select(conds, choices, default="fair")

# 4. Inspect all three groups
for cat in ["underpaid","fair","overpaid"]:
    print(f"\n=== {cat.upper()} ({cat=='fair' and 'close to predicted' or ''}) ===")
    display(
      performance
        .loc[performance["pay_category_dt"]==cat, 
             ["Player","Salary","salary_pred","residual"]]
        .sort_values(by="residual", ascending=(cat!="overpaid"))
        .head(4)
    )


Test RMSE: 12067388
Test R²:   0.013

=== UNDERPAID () ===


,Player,Salary,salary_pred,residual
400,Tyrese Haliburton,4215120.0,1.746314e+07,-1.324802e+07
224,Justin Holiday,6292440.0,1.746314e+07,-1.117070e+07
128,Eric Bledsoe,1300000.0,1.093189e+07,-9.631890e+06
21,Austin Reaves,1563518.0,1.093189e+07,-9.368372e+06



=== FAIR (close to predicted) ===


,Player,Salary,salary_pred,residual
315,Norman Powell,16758621.0,1.746314e+07,-704515.090909
105,Derrick White,16892857.0,1.746314e+07,-570279.090909
322,P.J. Tucker,10490000.0,1.093189e+07,-441890.153333
45,Cade Cunningham,10552800.0,1.093189e+07,-379090.153333



=== OVERPAID () ===


,Player,Salary,salary_pred,residual
363,Stephen Curry,48070014.0,1.093189e+07,3.713812e+07
351,Russell Westbrook,47063478.0,1.093189e+07,3.613159e+07
269,LeBron James,44474988.0,1.093189e+07,3.354310e+07
245,Kevin Durant,44119845.0,1.093189e+07,3.318795e+07


### Another way (performance-index)

In [49]:
from scipy.stats import zscore

# compute z‑scores column‑wise
zs = performance[featured_columns].apply(zscore)
zs.columns = [f"z_{c}" for c in featured_columns]
performance = performance.join(zs)
performance["perf_index"] = performance[[f"z_{c}" for c in featured_columns]].sum(axis=1)

# Compute cluster medians for perf_index and Salary
cluster_stats = performance.groupby("cluster").agg(
    med_perf   = ("perf_index", "median"),
    med_salary = ("Salary",      "median")
)

# Join those stats back onto each player
performance = performance.join(cluster_stats, on="cluster")

# Classify pay category:
conds = [
    # under‑paid: you perform ≥ your cluster’s median but cost ≤ its median
    (performance["perf_index"] >= performance["med_perf"]) &
    (performance["Salary"]     <= performance["med_salary"]),

    # over‑paid: you perform ≤ your cluster’s median but cost ≥ its median
    (performance["perf_index"] <= performance["med_perf"]) &
    (performance["Salary"]     >= performance["med_salary"])
]
choices = ["underpaid", "overpaid"]

performance["pay_category"] = np.select(conds, choices, default="fair")

# Quick look at your three groups
for cat in ["underpaid","fair","overpaid"]:
    print(f"\n {cat.upper()} PLAYERS: ")
    display(
        performance
          .loc[performance["pay_category"]==cat, 
               ["Player","cluster","perf_index","Salary"]]
          .sort_values(["perf_index","Salary"], ascending=[False, True])
          .head(3)
    )


 UNDERPAID PLAYERS: 


,Player,cluster,perf_index,Salary
315,Norman Powell,1,11.670872,16758621.0
358,Seth Curry,1,10.882929,8496653.0
400,Tyrese Haliburton,1,9.724256,4215120.0



 FAIR PLAYERS: 


,Player,cluster,perf_index,Salary
44,CJ McCollum,1,13.247637,33333333.0
42,Buddy Hield,1,11.901774,21177750.0
181,James Harden,1,11.258738,33000000.0



 OVERPAID PLAYERS: 


,Player,cluster,perf_index,Salary
361,Spencer Dinwiddie,1,7.862407,20171427.0
51,Caris LeVert,1,7.713922,18796296.0
105,Derrick White,1,6.931746,16892857.0


### *Another way to measure a player's performance: PER (Player Efficiency Rating)

In [50]:
# Reload the dataset
performance = pd.read_csv('../data/NBA_Perf_22.csv',encoding='latin1')
performance["Player"] = performance["Player"].str.replace("?", "ć")
# Calculate the PER with the PER formula
performance['PER']= (performance['FG']*85.910+performance['STL']*53.897+performance['3P']*51.757+performance['FT']*46.845+performance['BLK']*39.190+performance['ORB']*39.190+performance['DRB']*14.707+performance['AST']*34.677-(performance['FGA']-performance['FG'])*39.190-(performance['FTA']-performance['FT'])*20.091-performance['TOV']*53.897-performance['PF']*17.174)*(1/performance['MP'])
performance['PER'] = performance['PER'].round(2)
performance['PER'] = performance['PER'].replace([np.inf, -np.inf], np.nan)
performance['PER'] = performance['PER'].fillna(0)
performance['PER'] = performance['PER'].astype(float)
def filter_played(df):
    df = df[df['MP'] > 10]
    df = df[df['G']>20]
    return df
performance = filter_played(performance)
top_10_players = performance.sort_values(by='PER', ascending=False).head(10)
print(top_10_players[['Player', 'PER']])

                    Player    PER
391           Nikola Jokić  38.48
15   Giannis Antetokounmpo  37.84
206            Joel Embiid  35.79
368           LeBron James  30.96
198           Kevin Durant  30.52
808             Trae Young  30.17
526              Ja Morant  29.94
178            Luka Donćić  29.48
736     Karl-Anthony Towns  29.00
160          Anthony Davis  28.17
